## 字符串输出解析器StrOutputParser 没有get_format_instructions()

In [1]:
from langchain_classic.output_parsers import DatetimeOutputParser
## 获取大模型
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import os
import dotenv


dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    model="deepseek-chat",
)

## 调用大模型
response=llm.invoke("hi")
##<class 'langchain_core.messages.ai.AIMessage'>
print(type(response))

## 获取字符串的输出结果
##方式1：自己调用.content
print(response.content)

##方式2：使用StrOutputParser
##<class 'langchain_core.messages.base.TextAccessor'>
parser=StrOutputParser()
response1=parser.invoke(response)
print(response1)
print(type(response1))



<class 'langchain_core.messages.ai.AIMessage'>
你好！👋 很高兴见到你！

我是DeepSeek，一个AI助手，由深度求索公司创造。我可以帮你解答问题、进行对话、协助处理各种任务。

有什么我可以帮助你的吗？无论是学习、工作、生活上的问题，还是想聊聊天，我都很乐意和你交流！😊
你好！👋 很高兴见到你！

我是DeepSeek，一个AI助手，由深度求索公司创造。我可以帮你解答问题、进行对话、协助处理各种任务。

有什么我可以帮助你的吗？无论是学习、工作、生活上的问题，还是想聊聊天，我都很乐意和你交流！😊
<class 'langchain_core.messages.base.TextAccessor'>


## Json输出解析器JsonOutputParser
## 方式1：

In [3]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.prompts import ChatPromptTemplate
chat_prompt_template=ChatPromptTemplate.from_messages(
    [
        ("system","你是一个AI助手，名字叫{role}"),
        ("human","{question}")
    ]
)

prompt=chat_prompt_template.invoke({
    "role":"小丽",
    "question":"人工智能的英语怎么说？问题用q表示，答案用a表示，返回一个JSON格式的数据"
})
response=llm.invoke(prompt)

##获取一个JSONOutputParser的实例
parser=JsonOutputParser()
response1=parser.invoke(response)
print(response1)
print(type(response1))


{'q': '人工智能的英语怎么说？', 'a': '人工智能的英语是 "Artificial Intelligence"，通常简称为 AI。'}
<class 'dict'>


### 方式2：

In [5]:
parser=JsonOutputParser()
print(parser.get_format_instructions())

Return a JSON object.


### 举例2：管道符引入

In [7]:
from langchain_core.prompts import PromptTemplate

parser=JsonOutputParser()
Ques="人工智能的英语怎么说？"
prompt_template=PromptTemplate.from_template(
    template="回答用户的查询，回答的格式为{format_instructions},问题为{question}",
    partial_variables={"format_instructions":parser.get_format_instructions()}
)
##写法一：
# prompt=prompt_template.invoke({"question":Ques})
# response=llm.invoke(prompt)
# json_result=parser.invoke(response)
# print(json_result)

##写法二：
chain=prompt_template | llm | parser
print(chain.invoke({"question":Ques}))

{'query': '人工智能的英语怎么说？', 'answer': '人工智能的英语是 "Artificial Intelligence"，通常缩写为 "AI"。'}


### 3.XMLOutputParser XML输出解析器的使用
#### 举例1：

In [8]:
actor_query="周星驰的介绍"
response=llm.invoke(f"请生成{actor_query},生成的内容放在<movie></movie>标签中")
print(response.content)

<movie>
周星驰，1962年6月22日出生于香港，是华语影坛的标志性人物，集演员、导演、编剧、制片人于一身，被影迷尊称为“星爷”。他是无厘头喜剧风格的宗师级代表，其作品深刻影响了数代观众，成为华人流行文化的重要组成部分。

**核心风格与成就：**
周星驰的电影以“无厘头”喜剧著称，擅长将夸张的表演、荒诞的情节、草根小人物的悲欢离合与深刻的社会讽刺融为一体。他的电影表面是嬉笑怒骂，内核却往往充满对梦想、爱情、奋斗的温情与哲思，形成了“笑中带泪”的独特艺术效果。

**重要演艺阶段：**
1.  **电视与电影初期（1980年代）**：毕业于TVB艺员训练班，曾主持儿童节目《430穿梭机》，后参演多部电视剧。在电影圈初期多跑龙套或饰演配角，直至1988年凭《霹雳先锋》获得金马奖最佳男配角，开始崭露头角。
2.  **无厘头喜剧之王（1990年代）**：这是周星驰的黄金时代，他主演了一系列打破票房纪录、成为文化现象的经典作品，确立了其无可替代的喜剧之王地位。代表作包括：
    *   《赌圣》（1990）：奠定其票房号召力。
    *   《逃学威龙》系列（1991）：校园喜剧经典。
    *   《审死官》（1992）：展现喜剧中的批判锋芒。
    *   《唐伯虎点秋香》（1993）：古装喜剧巅峰之作。
    *   《大话西游》系列（1995）：初期票房未达预期，但日后凭借其跨越时空的凄美爱情与深刻内涵，在网络时代被重新发掘，封为神作，影响深远。
    *   《食神》（1996）、《喜剧之王》（1999）等，喜剧风格逐渐融入更多个人作者色彩与人生感悟。
3.  **自导自演的作者时期（2000年代以后）**：转型导演后，他对电影的控制力更强，作品在保持喜剧外壳的同时，视觉特效和人文思考更为突出。
    *   《少林足球》（2001）：开创香港电影特效新时代，荣获香港电影金像奖最佳影片、最佳导演等多项大奖。
    *   《功夫》（2004）：将其儿时武侠梦、特效技术与市井情怀结合，被誉为集大成之作，享誉国际。
    *   《长江7号》（2008）：转向科幻温情题材。
    *   《西游·降魔篇》（2013）、《美人鱼》（2016，创下当时中国影史票房纪录）、《新喜剧之王》（2019）等，持续探索不同题材。

**影响与评价：**


#### 举例二：

In [9]:
from langchain_core.output_parsers import XMLOutputParser

parser=XMLOutputParser()
print(parser.get_format_instructions())


The output should be formatted as a XML file.
1. Output should conform to the tags below.
2. If tags are not given, make them on your own.
3. Remember to always open and close all the tags.

As an example, for the tags ["foo", "bar", "baz"]:
1. String "<foo>
   <bar>
      <baz></baz>
   </bar>
</foo>" is a well-formatted instance of the schema.
2. String "<foo>
   <bar>
   </foo>" is a badly-formatted instance.
3. String "<foo>
   <tag>
   </tag>
</foo>" is a badly-formatted instance.

Here are the output tags:
```
None
```


In [11]:
## 使用结构XMLOutputParser
actor_query="生成邓紫棋的歌唱生涯"
parser=XMLOutputParser()
prompt_template=PromptTemplate.from_template(
    template="用户的问题为：{query},使用的格式为:{format}",
    partial_variables={"format":parser.get_format_instructions()}
)
prompt=prompt_template.invoke({"query":actor_query})
response=llm.invoke(prompt)
print(response.content)

<?xml version="1.0" encoding="UTF-8"?>
<歌唱生涯>
    <歌手>
        <姓名>邓紫棋</姓名>
        <英文名>G.E.M.</英文名>
        <本名>邓诗颖</本名>
        <出生日期>1991年8月16日</出生日期>
    </歌手>
    <生涯概述>
        <阶段>
            <时期>早期生涯与香港出道</时期>
            <起始年份>2008</起始年份>
            <描述>16岁时于香港出道，发行首张EP《G.E.M.》，凭借出众唱腔获得关注，夺得2008年叱咤乐坛流行榜颁奖典礼“叱咤乐坛生力军女歌手金奖”。</描述>
        </阶段>
        <阶段>
            <时期>巩固地位与突破</时期>
            <起始年份>2009-2013</起始年份>
            <描述>发行多张畅销专辑如《18...》、《MySecret》，并举办首次个人巡回演唱会。此阶段确立其香港乐坛小天后的地位，代表作包括《A.I.N.Y.》、《泡沫》等。</描述>
        </阶段>
        <阶段>
            <时期>进军内地与现象级走红</时期>
            <起始年份>2014</起始年份>
            <描述>参加中国内地综艺节目《我是歌手》第二季，以强劲实力和代表作《泡沫》等歌曲迅速红遍全国，成为华语乐坛焦点，打开庞大的内地市场。</描述>
        </阶段>
        <阶段>
            <时期>全面创作与国际化</时期>
            <起始年份>2015-2018</起始年份>
            <描述>发行全创作专辑《新的心跳》，展现其词曲创作、制作的全能才华。举办“Queen of Hearts”世界巡回演唱会，影响力扩展至国际。歌曲《光年之外》成为YouTube上首支破亿华语女歌手MV。</描述>
        </阶段>
        <阶段>
            <时期>独立发展与音乐探索</时期>
            <起始年份>2019至今<

### 4.列表解析器CSV  CommaSeparatedListOutputParser

In [13]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
output_praser = CommaSeparatedListOutputParser()
print(output_praser.get_format_instructions())

messages="大象，星星，小丽"
response=output_praser.parse(messages)
print(response)
print(type(response))

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['大象，星星，小丽']
<class 'list'>


### 5.日期时间解析器 DatetimeOutputParser

In [14]:
from langchain_classic.output_parsers import DatetimeOutputParser
parser=DatetimeOutputParser()
print(parser.get_format_instructions())

Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.

Examples: 2023-07-04T14:30:00.000000Z, 1999-12-31T23:59:59.999999Z, 2025-01-01T00:00:00.000000Z

Return ONLY this string, no other words!


In [18]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template=ChatPromptTemplate.from_messages(
    [
        ("system","{format}"),
        ("human","{question}")
    ]
)
parser=DatetimeOutputParser()
prompt=prompt_template.invoke({
    "format":parser.get_format_instructions(),"question":"中华人民共和国是什么时候成立的？"
})
response=llm.invoke(prompt)
print(parser.invoke(response))

1949-10-01 00:00:00
